# 4. NOMAD & NOMAD Oasis: Anatomy of a Research-Data Repository

Notebooks 1 and 2 gave you the ingredients — FAIR metadata and a shared
ontology (EMMO) to make that metadata unambiguous. This notebook shows what
those ingredients look like once they are put together into an actual
running **repository**: a piece of software whose entire job is to store,
validate, and let you search materials data at scale. We use **NOMAD** and
**NOMAD Oasis** as the worked example because they are a real, EMMO-aware
infrastructure used across European materials science — not a hypothetical
system.

This notebook is deliberately **read-only with respect to any live
server**: everything you run here is local Python that mimics NOMAD's data
model on a small scale, so the ideas are concrete and testable without
needing an account, an API key, or a working internet connection. Notebook 5
then has you apply the same pattern to build a small dataset of your own.

**Topics**
1. What NOMAD and NOMAD Oasis are (and how they relate)
2. The core data model: Upload → Entry → raw data / archive / metadata
3. Schemas: NOMAD's Metainfo, and ELN schemas for experimentalists
4. Where EMMO fits into a NOMAD entry
5. Hands-on: building a NOMAD-style archive entry in Python
6. Hands-on: a schema validator and a local "search" over entries

## 4.1 What Is NOMAD?

**NOMAD** (`nomad-lab.eu`) is an open-source, FAIR data infrastructure for
materials science, developed by the **FAIRmat** consortium (part of
Germany's National Research Data Infrastructure, NFDI) together with an
international community of contributors. It began as a repository for
computational materials data (parsing the output of dozens of DFT and
simulation codes into one common structure) and has since grown
**Electronic Lab Notebook (ELN)** support for experimental data too —
synthesis steps, characterisation measurements, and instrument metadata —
which is the part most directly relevant to this course.

**NOMAD Oasis** is the same underlying software, packaged so that an
institution or research group can run their **own private instance** —
typically for data that is not yet ready to be public (unpublished results,
an ongoing PhD project, embargoed industry collaboration data). An Oasis
still speaks NOMAD's data model and file formats, and can optionally
**publish** individual entries to the central, public NOMAD repository once
they are ready to be shared — so a lab's private working data and the
public scientific record use exactly the same structure throughout, instead
of being reformatted at the point of publication.

## 4.2 The Core Data Model: Upload → Entry

Everything in NOMAD starts with an **Upload**: a batch of files a user adds
to the system (raw instrument output, a lab-notebook YAML file, a folder of
DFT output files — whatever was actually produced). NOMAD's processing
pipeline then splits an upload into one or more **Entries** — the atomic,
citable unit in NOMAD, roughly "one measurement" or "one calculation." Each
entry carries three distinct layers of information:

| Layer | What it is | Analogy from Notebook 1 |
|---|---|---|
| **Raw data** | The original files exactly as uploaded, untouched | The instrument's raw export file |
| **Archive** | The raw data *parsed* into NOMAD's structured schema (Section 4.3) | The `cathode_record` dict, but machine-validated |
| **Metadata** | Searchable summary fields (upload ID, entry ID, authors, dates, references) | The F/A/R fields from the `fair_check` function |

This three-layer split is the same idea Notebook 1 introduced — separate
the *data* from the *description of the data* — just implemented at
production scale, with the parsing step automated so it does not depend on
every researcher manually filling in a form correctly.

## 4.3 Schemas: The Metainfo, and ELN Schemas for Experimentalists

An **archive** is not a free-form JSON blob — it must conform to a
**schema**, so that "synthesis temperature" always ends up in the same
place, with the same name and unit, across every entry that uses it. NOMAD
calls its schema system the **Metainfo**: definitions of **Sections**
(nested groups, like a Python class) containing **Quantities** (typed,
unit-aware fields) and **SubSections** (nested sections). If that sounds
like the metadata dictionaries from Notebook 1 — it is; a schema is simply
metadata *about the structure metadata is allowed to take*.

Two ways to define a schema, matching two kinds of user:

- **Python-defined schemas** (`nomad.metainfo`): used for computational data
  parsers, where a developer writes Python classes describing e.g. a DFT
  calculation's structure once, and every parser for that method reuses it.
- **ELN schemas** (YAML files, conventionally named `*.archive.yaml`): used
  by experimentalists to describe their own workflow — a synthesis
  procedure, a measurement type — *without writing Python*. This is the
  format most directly useful to a materials scientist setting up their own
  Oasis for lab work, and the one Notebook 5's capstone mirrors.

## 4.4 Where EMMO Fits Into a NOMAD Entry

Recall from Notebook 2 that EMMO's job is to make sure "temperature" means
the same formal thing everywhere. NOMAD schemas support attaching an
ontology **concept IRI** to individual quantities as an annotation — so a
quantity literally named `synthesis_temperature` in one lab's ELN schema
and `T_anneal` in another's can both be marked as instances of the *same*
EMMO concept, making them comparable by software even though their column
names disagree. This is the practical payoff of Notebook 2's abstract point
about `SpecificHeatCapacity` being a `Sign`: NOMAD entries carry exactly
this kind of sign-to-concept link, at the level of individual data
fields.

## 4.5 Hands-on: Building a NOMAD-Style Archive Entry

The cell below is an **illustrative sketch** of the pattern a NOMAD archive
follows for an ELN entry — simplified, and with shortened field names, to
keep the structure visible rather than reproducing any specific real
schema line-for-line. Compare it directly to `cathode_record` from
Notebook 1 — and to the SQL rows and Mongo documents built for the same
batches in Notebook 3 — same information, now split explicitly into the
three layers from Section 4.2, and with an EMMO IRI attached to two of
the quantities the way Section 4.4 described.

In [ ]:
import json

nomad_style_entry = {
    # ── metadata layer: searchable summary fields ─────────────────────────────
    'metadata': {
        'upload_id':  'upload_2025_01_cathodes',
        'entry_id':   'entry_LiFePO4_batch07',
        'upload_create_time': '2025-01-15T09:12:00Z',
        'authors':    ['A. Lindqvist'],
        'references': [],
        'license':    'CC-BY-4.0',
    },
    # ── archive layer: parsed, schema-conformant structure ────────────────────
    'archive': {
        'data': {
            'm_def': 'CathodeSynthesisEntry',            # which schema this entry follows
            'name': 'LiFePO4_batch_07',
            'synthesis': {
                'method': 'solid-state reaction',
                'steps': [
                    {'name': 'mixing',     'duration_h': 2},
                    {'name': 'calcination_1', 'temperature_C': 350, 'duration_h': 4},
                    {'name': 'calcination_2', 'temperature_C': 750, 'duration_h': 6,
                     'quantity_kind_iri': 'emmo:ThermodynamicTemperature'},
                ],
                'atmosphere': 'Ar',
            },
            'results': {
                'capacity_mAh_g': 161.4,
                'quantity_kind_iri': 'emmo:ElectricCharge',
                'coulombic_efficiency': 0.94,
            },
        },
    },
    # ── raw data layer: pointers to the original files, not reproduced here ───
    'raw_files': ['furnace_log_batch07.csv', 'cycler_export_batch07.txt'],
}

print(json.dumps(nomad_style_entry, indent=2))

## 4.6 A Minimal Schema Validator

A NOMAD schema does more than organise data — it *rejects* entries that
don't conform, the same way a type checker rejects code with the wrong
types. We can mimic this in plain Python with a small validator function
that checks required keys and simple type constraints, the same spirit as
`fair_check` from Notebook 1 but checking *structure* instead of FAIR
fields.

In [ ]:
def validate_entry(entry, schema):
    """Very small structural validator: checks required keys and types
    against a nested schema dict of the form {key: type_or_nested_schema}."""
    errors = []

    def _check(data, schema, path=''):
        for key, expected in schema.items():
            full_path = f'{path}.{key}' if path else key
            if key not in data:
                errors.append(f'MISSING: {full_path}')
                continue
            if isinstance(expected, dict):
                if isinstance(data[key], dict):
                    _check(data[key], expected, full_path)
                else:
                    errors.append(f'EXPECTED nested object at {full_path}')
            elif not isinstance(data[key], expected):
                errors.append(
                    f'WRONG TYPE at {full_path}: expected {expected.__name__}, '
                    f'got {type(data[key]).__name__}'
                )

    _check(entry, schema)
    return errors


cathode_schema = {
    'name': str,
    'synthesis': {
        'method': str,
        'atmosphere': str,
    },
    'results': {
        'capacity_mAh_g': (int, float),
        'coulombic_efficiency': (int, float),
    },
}

errors = validate_entry(nomad_style_entry['archive']['data'], cathode_schema)
print('Validation errors:', errors if errors else 'none — entry conforms to schema')

# Now break it on purpose and re-validate
broken_entry = json.loads(json.dumps(nomad_style_entry['archive']['data']))  # deep copy
del broken_entry['results']['capacity_mAh_g']
broken_entry['synthesis']['method'] = 123   # wrong type

print('Validation errors on broken entry:', validate_entry(broken_entry, cathode_schema))

## 4.7 A Local Stand-in for a NOMAD Search Query

The real NOMAD offers both a graphical search interface and a REST API
(`api/v1/entries/query` on `nomad-lab.eu`, or your Oasis's own URL) so that
software can filter entries by any metainfo quantity — "find every entry
with `synthesis.method == 'solid-state reaction'` and
`results.capacity_mAh_g > 150`," across potentially millions of entries. The
exact request syntax is documented on NOMAD's own site and evolves between
versions, so it is not reproduced here — but the underlying *idea* is just
filtering a collection of structured records, which you already know how to
do with plain Python. The cell below builds a small local collection of
entries and filters it exactly the way a remote query would, so the logic
is identical even though the data now lives in a Python list instead of on
a server.

In [ ]:
def make_entry(name, method, temp_C, capacity, efficiency):
    e = json.loads(json.dumps(nomad_style_entry))    # start from the template, deep copy
    d = e['archive']['data']
    d['name'] = name
    d['synthesis']['method'] = method
    d['synthesis']['steps'][-1]['temperature_C'] = temp_C
    d['results']['capacity_mAh_g'] = capacity
    d['results']['coulombic_efficiency'] = efficiency
    e['metadata']['entry_id'] = f'entry_{name}'
    return e


local_repository = [
    make_entry('batch_05', 'solid-state reaction', 700, 148.2, 0.91),
    make_entry('batch_06', 'solid-state reaction', 725, 155.9, 0.93),
    make_entry('batch_07', 'solid-state reaction', 750, 161.4, 0.94),
    make_entry('batch_08', 'sol-gel',              650, 172.1, 0.95),
    make_entry('batch_09', 'sol-gel',              680, 168.7, 0.92),
]

# ── A local 'query': mimics filtering entries via a NOMAD-style API call ──────
def query_entries(entries, method=None, min_capacity=None):
    results = entries
    if method is not None:
        results = [e for e in results if e['archive']['data']['synthesis']['method'] == method]
    if min_capacity is not None:
        results = [e for e in results
                   if e['archive']['data']['results']['capacity_mAh_g'] >= min_capacity]
    return results


hits = query_entries(local_repository, method='sol-gel', min_capacity=170)
print(f'{len(hits)} entr{"y" if len(hits)==1 else "ies"} matched:')
for e in hits:
    d = e['archive']['data']
    print(f"  {d['name']:<10} capacity={d['results']['capacity_mAh_g']} mAh/g  "
          f"method={d['synthesis']['method']}")

This is the essence of what a repository buys you over a folder of
spreadsheets: because every entry conforms to the same validated schema
(Section 4.6), a query like `query_entries` can run correctly over five
entries or five million, and over your data as well as a collaborator's —
provided both of you used the same schema, or schemas that share EMMO
concepts (Section 4.4) so a query can be translated between them.

---
## Exercises

1. **Extend the local repository**: Add three more entries to
   `local_repository` (invent plausible values) and write a `query_entries`
   call that finds all entries with `coulombic_efficiency >= 0.93`
   regardless of method. You will need to add a new parameter to
   `query_entries` to support this.

2. **Schema evolution**: Add a new required field `operator` (a string)
   to `cathode_schema`, at the top level next to `name`. Re-run the
   validator against `nomad_style_entry['archive']['data']` — what error do
   you get, and what does that tell you about what happens in a real
   repository when a schema changes after data has already been uploaded?

3. **Design question**: `validate_entry` only checks that required fields
   are *present with the right type* — it does not check that
   `coulombic_efficiency` is between 0 and 1, or that `capacity_mAh_g` is
   positive. Sketch (in words or code) how you would extend the schema
   format to support a numeric range constraint, and add it for one field.